# HYPER-3, 2 — What the trial identifies, and what it does not

Randomization is the strongest identification argument there is, and it is *narrow*.
It licenses one quantity — the effect of the **assigned** dose on the outcome, in the
population that was randomized — and it licenses that quantity only for analyses that
do not condition on anything downstream of the assignment.

Three things in HYPER-3 sit downstream of the assignment: **adherence**, **dropout**,
and the pressure readings themselves. Two of the three are tempting to condition on.
This notebook writes the graph down, asks `axiom.identify` what it licenses, and
measures what each temptation costs on the trial's own data.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import hyper3 as h
from axiom.core import BASES, D, Covariate, Intervention, Outcome, Population, TimeWindow, Treatment
from axiom.estimands import Estimand, Level, Quantity, TransferPlan, standard_estimands
from axiom.identify import (
    CausalGraph, IdentificationVerdict, TransportVerdict, adjustment_sets, assign_roles,
    backdoor_admissible, directly_transportable, identify, minimal_adjustment_sets, ols,
    requires_unmeasured, roles, s_admissible, s_admissible_sets, selection_diagram,
    transport_verdict, trivially_transportable,
)

from axiom.display import enable, table

enable();  # every axiom result renders itself from here on

BASES.declare("mass", symbol="M")
trial = h.trial(seed=20260821)
final = h.look_frame(trial, h.TRIAL_WEEKS, endpoint="primary")
print(f"{len(final)} units reached the primary window (weeks "
      f"{h.WINDOW['primary'][0]}–{h.WINDOW['primary'][1]}) out of {trial.n_units} randomized")

## 1. The graph

`age` causes baseline pressure, causes the response, and causes dropout. `dose` is
**randomized**, so it has no parents at all — that single fact is the whole
identification argument. Adherence sits between dose and outcome. Dropout is caused by
how a unit is doing, so it is a descendant of the outcome. `frailty` is the unmeasured
thing that drives both the outcome and dropout, and it is what makes dropout dangerous.

In [ ]:
graph = CausalGraph.from_edges(
    """
    age -> baseline; age -> sbp; age -> dropout;
    baseline -> sbp; baseline -> dropout;
    dose -> adherence; dose -> sbp;
    adherence -> sbp;
    sbp -> dropout;
    frailty -> sbp; frailty -> dropout
    """,
    unmeasured=("frailty",),
    name="hyper3",
)
print("nodes     :", sorted(graph.nodes))
print("measured  :", sorted(graph.measured))
print("hash      :", graph.content_hash()[:16])
print("\nrole of each node in the dose -> sbp question:")
table(
    [[node, role] for node, role in sorted(roles(graph, "dose", "sbp").items())],
    headers=("node", "role"),
)

In [ ]:
POSITION = {"age": (0.0, 1.0), "baseline": (1.0, 1.0), "dose": (0.0, 0.0),
            "adherence": (1.0, 0.0), "sbp": (2.0, 0.5), "dropout": (3.0, 0.5),
            "frailty": (2.0, 1.4)}
ROLE_COLOR = {"treatment": "#2f7fd1", "outcome": "#d1483f", "mediator": "#8a63c4",
              "descendant_of_outcome": "#c9a227", "neutral": "#5b6472"}


def draw_graph(g: CausalGraph, title: str, highlight: set[str] = frozenset()) -> go.Figure:
    """A DAG as annotated arrows — nodes coloured by the role `identify` gives them."""
    role_of = {k: str(v) for k, v in roles(g, "dose", "sbp").items()}
    fig = h.figure(title, "", "", height=380)
    for src, dst in g.edges:
        x0, y0 = POSITION[src]
        x1, y1 = POSITION[dst]
        dx, dy = x1 - x0, y1 - y0
        norm = float(np.hypot(dx, dy))
        pad = 0.16
        fig.add_annotation(
            x=x1 - pad * dx / norm, y=y1 - pad * dy / norm,
            ax=x0 + pad * dx / norm, ay=y0 + pad * dy / norm,
            xref="x", yref="y", axref="x", ayref="y", showarrow=True, arrowhead=2,
            arrowwidth=1.6, arrowcolor="rgba(70,70,70,0.55)",
        )
    for node, (x, y) in POSITION.items():
        if node not in g.nodes:
            continue
        colour = ROLE_COLOR.get(role_of.get(node, "neutral"), "#5b6472")
        latent = node in g.unmeasured
        fig.add_trace(go.Scatter(
            x=[x], y=[y], mode="markers+text", text=[node], textposition="middle center",
            textfont={"size": 11, "color": "white" if not latent else "#333"},
            marker={"size": 62, "color": "rgba(255,255,255,0.9)" if latent else colour,
                    "line": {"width": 3 if node in highlight else 1.6,
                             "color": "#111" if node in highlight else colour},
                    "symbol": "circle"},
            showlegend=False, hovertext=role_of.get(node, ""), hoverinfo="text"))
    fig.update_xaxes(visible=False, range=[-0.5, 3.5])
    fig.update_yaxes(visible=False, range=[-0.5, 1.9])
    return fig


draw_graph(graph, "HYPER-3: dose is randomized, so it has no parents")

## 2. What randomization buys

`identify` returns a verdict, a route, and the set to adjust for. For a randomized
treatment the answer is the empty set: nothing needs adjusting, because nothing causes
the dose.

In [ ]:
verdict = identify(graph, "dose", "sbp")
assert isinstance(verdict, IdentificationVerdict)
print("status         :", verdict.verdict.status)
print("route          :", verdict.route)
print("adjustment set :", verdict.adjustment_set or "{} (empty — dose has no parents)")
print("needs unmeasured:", requires_unmeasured(graph, "dose", "sbp"))
table(
    [[a.name, a.statement] for a in verdict.verdict.assumptions],
    headers=("assumption", "statement"),
)

print("\nminimal admissible sets:", [sorted(z) or "{}" for z in minimal_adjustment_sets(graph, "dose", "sbp")])
print("every admissible set    :", [sorted(z) or "{}" for z in adjustment_sets(graph, "dose", "sbp")])

`age` and `baseline` appear in admissible sets not because they are needed but because
adjusting for a pre-treatment cause of the outcome is *harmless* and, as notebook 1
showed, worth several mmHg of precision. `adherence` never appears, and that is the
point of the next section.

## 3. The first temptation: adjust for adherence

Adherence is measured, it is strongly related to the outcome, and adjusting for it
looks like it should sharpen the estimate. `assign_roles` says what it actually is.

In [ ]:
assignment = assign_roles(graph, "dose", "sbp")
by_role: dict[str, list[str]] = {}
for node, role in assignment.roles.items():
    by_role.setdefault(str(role), []).append(node)
table(
    [[role, str(sorted(nodes))] for role, nodes in sorted(by_role.items())],
    headers=("role", "nodes"),
)
admissible = adjustment_sets(graph, "dose", "sbp")
print("\nis {adherence} admissible?", frozenset({"adherence"}) in admissible)
print("is {baseline} admissible? ", frozenset({"baseline"}) in admissible)

In [ ]:
rows = []
for arm in h.ARMS[1:]:
    frame = h.contrast_frame(final, arm)
    itt = ols(frame, "change", "treated", h.ancova_covariates(None))
    conditioned = ols(frame, "change", "treated", h.ancova_covariates(None) + ["adherence"])
    rows.append({"arm": h.ARM_LABEL[arm],
                 "intent_to_treat": itt.estimate, "itt_se": itt.se,
                 "adjusted_for_adherence": conditioned.estimate, "adj_se": conditioned.se,
                 "truth": h.intent_to_treat_contrast(h.DOSE[arm])})
mediator = pd.DataFrame(rows)
print("contrast against the standard of care, week 9–12 window (mmHg)")
print(mediator.round(2).to_string(index=False))

fig = h.figure("Adjusting for adherence removes part of the effect it was measuring",
               "", f"contrast ({h.OUTCOME_UNIT})", height=380, barmode="group")
fig.add_trace(go.Bar(x=mediator["arm"], y=mediator["intent_to_treat"], name="intent to treat",
                     marker_color="#2f7fd1",
                     error_y={"type": "data", "array": 1.96 * mediator["itt_se"]}))
fig.add_trace(go.Bar(x=mediator["arm"], y=mediator["adjusted_for_adherence"],
                     name="+ adherence (a mediator)", marker_color="#8a63c4",
                     error_y={"type": "data", "array": 1.96 * mediator["adj_se"]}))
fig.add_trace(go.Scatter(x=mediator["arm"], y=mediator["truth"], mode="markers", name="truth",
                         marker={"symbol": "line-ew-open", "size": 40, "line": {"width": 3, "color": "#111"}}))
fig.add_hline(y=0, line={"color": "rgba(0,0,0,0.45)"})
fig

The adherence-adjusted column is not a better estimate of the same thing; it is an
estimate of a *different* thing — the direct effect of assignment holding adherence
fixed — and it is not what a regulator, a prescriber or a patient is asking about. On
the 40 mg arm it moves the number by 1.3 mmHg, most of what is left of that arm's
effect, and it *widens* the interval rather than narrowing it: the adjustment that was
supposed to buy precision cost some.
`identify` refuses to call it admissible, which is the useful behaviour: the graph
already knew.

## 4. The second temptation: analyse the units who finished

Dropout is caused by the outcome and by unmeasured frailty. Restricting the analysis
to the units who completed conditions on that node — and conditioning on a collider is
what *destroys* the one thing randomization gave us. The graph says it in one line: the
randomized dose is independent of frailty, and stops being independent of it the moment
the analysis conditions on who stayed.

In [ ]:
print("dose ⟂ frailty            :", graph.d_separated({"dose"}, {"frailty"}))
print("dose ⟂ frailty | dropout  :", graph.d_separated({"dose"}, {"frailty"}, {"dropout"}))
print("\nis {} admissible?         ", backdoor_admissible(graph, "dose", "sbp", ()))
print("is {dropout} admissible?  ", backdoor_admissible(graph, "dose", "sbp", ("dropout",)))
print("role of dropout           :", assignment.roles["dropout"])
draw_graph(graph, "Conditioning on dropout opens a path through the unmeasured node",
           highlight={"dropout", "frailty"})

In [ ]:
completers = set(trial.visits[trial.visits["week"] == h.FOLLOW_UP_WEEKS]["unit"])
rows = []
for arm in h.ARMS[1:]:
    frame = h.contrast_frame(final, arm, stratum="age_51_plus")
    everyone = ols(frame, "change", "treated", h.ancova_covariates("age_51_plus"))
    finished = frame[frame["unit"].isin(completers)]
    survivors = ols(finished, "change", "treated", h.ancova_covariates("age_51_plus"))
    rows.append({"arm": h.ARM_LABEL[arm], "randomized": everyone.estimate, "randomized_se": everyone.se,
                 "n_randomized": everyone.n, "completers": survivors.estimate,
                 "completers_se": survivors.se, "n_completers": survivors.n,
                 "truth": h.intent_to_treat_contrast(h.DOSE[arm], "age_51_plus")})
selection = pd.DataFrame(rows)
print("oldest stratum, contrast against the standard of care (mmHg)")
print(selection.round(2).to_string(index=False))
selection["randomized_error"] = selection["randomized"] - selection["truth"]
selection["completers_error"] = selection["completers"] - selection["truth"]
print("\nerror against the truth (mmHg):")
print(selection[["arm", "randomized_error", "completers_error"]].round(2).to_string(index=False))

The completer analysis moves every arm, and — this is what makes it dangerous — **not
in a common direction**: it pulls the 10 mg estimate toward the truth and pushes the
20 mg and 40 mg estimates further away. There is no sign to correct for,
because the bias runs through a node nobody measured. That is the use of consulting the
graph before the data: it does not say how large the bias is, it says the quantity is
no longer identified. The trial's primary analysis therefore uses every randomized unit
that reached the window.
## 5. What the trial would have been without randomization

Worth writing down, because it is the counterfactual that makes randomization worth
its cost. Suppose the dose had been chosen by the treating physician on the basis of
how high the pressure already was. Then `baseline -> dose` exists, and identification
needs an adjustment set.

In [ ]:
observational = CausalGraph.from_edges(
    """
    age -> baseline; age -> sbp; age -> dropout; age -> dose;
    baseline -> sbp; baseline -> dropout; baseline -> dose;
    dose -> adherence; dose -> sbp; adherence -> sbp; sbp -> dropout;
    frailty -> sbp; frailty -> dropout
    """,
    unmeasured=("frailty",), name="hyper3_observational",
)
obs_verdict = identify(observational, "dose", "sbp")
print("status:", obs_verdict.verdict.status, "| route:", obs_verdict.route)
print("adjustment set:", sorted(obs_verdict.adjustment_set))
print("minimal sets  :", [sorted(z) or "{}" for z in minimal_adjustment_sets(observational, "dose", "sbp")])
print("\nrandomized trial needs to adjust for:", verdict.adjustment_set or "nothing")
print("the same question observationally needs:", sorted(obs_verdict.adjustment_set))
draw_graph(observational, "Without randomization, baseline is a confounder", highlight={"baseline", "dose"})

## 6. Transporting to the population that will take the drug

The trial randomized 25 % / 40 % / 35 % across the age bands. The population that will
be prescribed the drug is older: 15 % / 35 % / 50 %. Because the effect **differs by
age band** — which is the entire finding of this case study — the trial's population
average does not transport. `transport_verdict` on a selection diagram says what has
to be conditioned on for it to.

In [ ]:
# A selection diagram marks each node whose distribution or mechanism differs between
# the two populations with an S-node pointing into it. Here exactly one does: the age
# composition. Nothing else about the biology is claimed to differ.
across = graph.with_selection("age")
diagram = selection_diagram(across)
print("S-nodes materialised   :", sorted(set(diagram.nodes) - set(graph.nodes)))
transport = transport_verdict(across, "dose", "sbp")
assert isinstance(transport, TransportVerdict)
print("status                 :", transport.verdict.status)
print("directly transportable :", directly_transportable(across, "dose", "sbp"),
      " <- the source effect is NOT the target effect")
print("s-admissible set       :", sorted(transport.s_admissible_set))
print("is {age} s-admissible? :", s_admissible(across, "dose", "sbp", ["age"]))
print("every s-admissible set :", [sorted(z) or "{}" for z in s_admissible_sets(across, "dose", "sbp")])
print("transport formula      :", transport.formula)
table(
    [[a.name, a.statement] for a in transport.verdict.assumptions],
    headers=("needs", "statement"),
)
print("\ntrivially transportable:", trivially_transportable(across, "dose", "sbp"))
print("  true, and useless: it says the target could identify the effect from its own")
print("  randomized trial - which is the trial nobody is going to run a second time.")

In [ ]:
TARGET_SHARE = {"age_25_35": 0.15, "age_36_50": 0.35, "age_51_plus": 0.50}
rows = []
for arm in h.ARMS[1:]:
    by_stratum = {}
    for stratum in h.STRATA:
        frame = h.contrast_frame(final, arm, stratum=stratum)
        by_stratum[stratum] = ols(frame, "change", "treated", h.ancova_covariates(stratum))
    trial_average = sum(h.STRATUM_SHARE[s] * by_stratum[s].estimate for s in h.STRATA)
    target_average = sum(TARGET_SHARE[s] * by_stratum[s].estimate for s in h.STRATA)
    trial_se = float(np.sqrt(sum((h.STRATUM_SHARE[s] * by_stratum[s].se) ** 2 for s in h.STRATA)))
    target_se = float(np.sqrt(sum((TARGET_SHARE[s] * by_stratum[s].se) ** 2 for s in h.STRATA)))
    rows.append({"arm": h.ARM_LABEL[arm], "trial_population": trial_average, "trial_se": trial_se,
                 "target_population": target_average, "target_se": target_se})
transported = pd.DataFrame(rows)
print("standardized contrast (mmHg), same stratum-specific estimates, different weights")
print(transported.round(2).to_string(index=False))

fig = make_subplots(rows=1, cols=2, subplot_titles=("Age composition", "Standardized contrast"))
fig.add_trace(go.Bar(x=[h.STRATUM_LABEL[s] for s in h.STRATA],
                     y=[h.STRATUM_SHARE[s] for s in h.STRATA], name="trial",
                     marker_color="#5b6472"), row=1, col=1)
fig.add_trace(go.Bar(x=[h.STRATUM_LABEL[s] for s in h.STRATA],
                     y=[TARGET_SHARE[s] for s in h.STRATA], name="target population",
                     marker_color="#2f7fd1"), row=1, col=1)
fig.add_trace(go.Bar(x=transported["arm"], y=transported["trial_population"], name="trial",
                     marker_color="#5b6472", showlegend=False,
                     error_y={"type": "data", "array": 1.96 * transported["trial_se"]}), row=1, col=2)
fig.add_trace(go.Bar(x=transported["arm"], y=transported["target_population"],
                     name="target population", marker_color="#2f7fd1", showlegend=False,
                     error_y={"type": "data", "array": 1.96 * transported["target_se"]}), row=1, col=2)
fig.add_hline(y=0, line={"color": "rgba(0,0,0,0.45)"}, row=1, col=2)
fig.update_yaxes(title_text="share", row=1, col=1, gridcolor=h.GRID)
fig.update_yaxes(title_text=f"contrast ({h.OUTCOME_UNIT})", row=1, col=2, gridcolor=h.GRID)
fig.update_layout(height=400, template="plotly_white", barmode="group",
                  title="The same trial, read for two different populations",
                  legend={"orientation": "h", "y": 1.12, "x": 0.0},
                  margin={"l": 60, "r": 30, "t": 100, "b": 50})
fig

The 40 mg arm is a modest benefit in the trial's population and a clear net harm in
the older population that would actually receive it. Nothing about the estimates
changed — only the weights.

## 7. Writing the estimands down

`standard_estimands` builds the usual family for one treatment, outcome, population,
window and level. The conditional versions — one per stratum — are the same estimand
with a `conditioning` facet, and `transfer_to` says exactly what differs between two
of them and what would have to be assumed to move a number from one to the other.

In [ ]:
sbp = Outcome(name="sbp_change", dimension=D.outcome, unit=h.OUTCOME_UNIT, aggregation="mean")
drug = Treatment(name="dose", dimension=D.mass, unit=h.DOSE_UNIT)
trial_population = Population(name="hyper3_randomized", strata={"age_band": dict(h.STRATUM_SHARE)})
target_population = Population(name="prescribing_population", strata={"age_band": dict(TARGET_SHARE)})
window = TimeWindow(start=h.WINDOW["primary"][0], stop=h.WINDOW["primary"][1])
level = Level(unit="individual", interference="none")

registry = standard_estimands(treatment=drug, outcome=sbp, population=trial_population,
                              window=window, level=level, dose=40.0, reference_dose=0.0)
print("standard estimands:", registry.names())
primary = registry.get("contrast_at_dose")
print("\nprimary:", primary.name, "|", primary.quantity.kind,
      "| dose", primary.intervention.doses, "vs", primary.reference.doses)
print("dimension:", primary.dimension, "| hash:", primary.content_hash()[:16])

oldest = primary.model_copy(update={"name": "contrast_at_dose_age_51_plus",
                                    "conditioning": ("age_51_plus",)})
retargeted = primary.model_copy(update={"name": "contrast_at_dose_prescribing",
                                        "population": target_population})
for target in (oldest, retargeted):
    plan = primary.transfer_to(target)
    assert isinstance(plan, TransferPlan)
    print(f"\n{primary.name} -> {target.name}")
    print(f"  status {plan.status} | differing facets: {plan.differing}")
    table([[e.facet, e.statement] for e in plan.entries], headers=("facet", "what differs"))
    table(
        [[a.name, a.statement] for a in plan.assumptions],
        headers=("the transfer needs", "statement"),
    )

## What this notebook decided

- Randomization identifies the intent-to-treat contrast with an **empty** adjustment
  set. Age and baseline pressure are adjusted for precision, not for identification,
  and `adjustment_sets` says so.
- Adherence is a mediator. Conditioning on it changes the estimand — on the 40 mg arm
  by more than a mmHg, and it widens the interval too. `identify` never lists it as
  admissible, and that refusal is the whole value of writing the graph down.
- Dropout is a descendant of the outcome with an unmeasured common cause. Conditioning
  on it makes the randomized dose dependent on that unmeasured cause — `d_separated`
  says so — and on this trial's data a completers analysis is further from the truth on
  every arm, in inconsistent directions. The primary analysis uses every randomized
  unit that reached the window.
- The trial's population average does **not** transport to an older prescribing
  population, because the effect differs by the very variable the populations differ
  on. `transport_verdict` names `age` as the s-admissible set; re-standardizing turns
  a modest benefit at 40 mg into a net harm.
- Every one of these is a statement about a graph, made before any estimate was
  computed. Notebook 3 turns the licensed estimand into a size.